# Notebook 1: Ingest & Unify — Crop Disease Datasets (Pakistan 6-crop model)

Run top-to-bottom in Google Colab or Kaggle Notebooks. Nothing downloads to your local machine — everything lives in the Colab/Kaggle runtime and is persisted to Google Drive at the end.

Companion docs:
- Dataset strategy: `docs/superpowers/specs/2026-08-12-crop-disease-dataset-strategy-design.md`
- This notebook's plan: `docs/superpowers/plans/2026-08-12-notebook1-ingest-unify.md`

**Two dataset slugs below are flagged UNVERIFIED — check them on their Kaggle page before trusting the results:**
- `citrus_primary` — disease names in the original brief (Ash Weevil, Dry Root Rot) didn't match any Kaggle set found; using the closest Pakistan/Kinnow-relevant match instead.
- The "71-classes" multi-crop dataset from the original brief could not be found on Kaggle under that description — substituted with a confirmed 5-crop Pakistan-relevant set instead.

## Task 1: Environment setup + Kaggle auth

In [ ]:
!pip install -q kagglehub huggingface_hub datasets pillow pandas

In [ ]:
import os
from google.colab import userdata

# Kaggle: reads Colab secret named exactly KAGGLE_API_TOKEN.
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
import kagglehub
print("Kaggle auth token loaded:", bool(os.environ.get("KAGGLE_API_TOKEN")))

# Hugging Face: reads Colab secret named exactly HF_TOKEN.
# LeafNet (enalis/LeafNet) is a GATED dataset — a valid token alone is not
# enough. Visit https://huggingface.co/datasets/enalis/LeafNet while logged
# into HF and click "Agree and access repository" ONCE before running the
# download cell below, or this will keep 401'ing even with a correct token.
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
print("HF auth token loaded.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Task 2: Dataset registry config

In [ ]:
# Slugs found via web search 2026-08-12. Confidence noted per entry —
# verify on the dataset's Kaggle/HF page before large training runs,
# especially the two flagged UNVERIFIED.
DATASETS = {
    "wheat_primary": {
        "type": "kaggle", "slug": "khanaamer/wheat-leaf-disease-dataset",
        "crop": "wheat"},  # confidence: medium — title matches, class count (5) unconfirmed
    "cotton_primary": {
        "type": "kaggle", "slug": "seroshkarim/cotton-leaf-disease-dataset",
        "crop": "cotton"},  # confidence: high — 2137 orig + 7000 aug, 7 classes confirmed
    "rice_primary": {
        "type": "kaggle", "slug": "nirmalsankalana/rice-leaf-disease-image",
        "crop": "rice"},  # confidence: medium-high — 5,932 images confirmed, class list unconfirmed
    "sugarcane_primary": {
        "type": "kaggle", "slug": "akilesh253/sugarcane-plant-diseases-dataset",
        "crop": "sugarcane"},  # confidence: high — 19,926 images, 6 classes confirmed
    "citrus_primary": {
        "type": "kaggle", "slug": "myprojectdictionary/citrus-leaf-disease-image",
        "crop": "citrus"},  # UNVERIFIED — disease names don't match original brief, double check
    "mango_primary": {
        "type": "kaggle", "slug": "aryashah2k/mango-leaf-disease-dataset",
        "crop": "mango"},  # confidence: high — MangoLeafBD, 4,000 images, 8 classes confirmed
    "multi_crop_supplement": {
        "type": "kaggle", "slug": "jawadali1045/20k-multi-class-crop-disease-images",
        "crop": None},  # UNVERIFIED — substituted for the not-found "71-classes" dataset; covers wheat/maize/cotton/sugarcane/rice
    "leafnet_field": {
        "type": "hf", "slug": "enalis/LeafNet",
        "crop": None},  # confidence: high — 186k in-situ images, 22 species, confirmed
}

missing = [k for k, v in DATASETS.items() if v["slug"] == "FILL_ME"]
assert not missing, f"Fill in slugs before proceeding: {missing}"
print(f"{len(DATASETS)} datasets configured.")

## Task 3: Download all sources into runtime storage

In [ ]:
import kagglehub
from huggingface_hub import snapshot_download

RAW_PATHS = {}
FAILED = {}
for key, cfg in DATASETS.items():
    print(f"Downloading {key} ({cfg['slug']}) ...")
    try:
        if cfg["type"] == "kaggle":
            path = kagglehub.dataset_download(cfg["slug"])
        elif cfg["type"] == "hf":
            path = snapshot_download(repo_id=cfg["slug"], repo_type="dataset")
        else:
            raise ValueError(f"Unknown type for {key}: {cfg['type']}")
        RAW_PATHS[key] = path
        print(f"  -> {path}")
    except Exception as e:
        FAILED[key] = str(e)
        print(f"  FAILED: {e}")

print(f"\n{len(RAW_PATHS)}/{len(DATASETS)} downloaded.")
if FAILED:
    print("Failed keys (fix access/slug and re-run this cell to retry just these):")
    for k in FAILED:
        print(f"  - {k}")

# leafnet_field failing here is not blocking for this notebook — it only
# feeds Notebook 2's domain-pretrain step. The 6 crop + supplement datasets
# below are what Task 5 onward actually needs.
assert all(k in RAW_PATHS for k in DATASETS if k != "leafnet_field"), \
    f"A required (non-LeafNet) dataset failed to download: {[k for k in FAILED if k != 'leafnet_field']}"
print("All required crop datasets present — safe to continue to Task 4.")

If any download 404s here, that dataset's slug is wrong (most likely `citrus_primary` or `multi_crop_supplement`, the two flagged UNVERIFIED above) — go check the Kaggle page URL directly and fix the slug in Task 2.

## Task 4: Explore raw folder structure per dataset

In [ ]:
def print_tree(path, max_depth=2, max_entries=15):
    for root, dirs, filenames in os.walk(path):
        depth = root[len(path):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root) or root}/")
        for f in (filenames[:max_entries] if depth == max_depth else []):
            print(f"{indent}  {f}")
        if len(filenames) > max_entries and depth == max_depth:
            print(f"{indent}  ... ({len(filenames)} files total)")

for key, path in RAW_PATHS.items():
    print(f"\n=== {key} ({path}) ===")
    print_tree(path)

**Human step:** read the tree output above. For each dataset, note which subfolder names correspond to which disease label. Use that to fill `LABEL_MAPS` in Task 5 below — the exact subfolder names cannot be known before this cell has actually run against the real downloaded data.

## Task 5: Per-dataset normalizers → unified `crop/disease/` layout

In [ ]:
import shutil

UNIFIED_ROOT = "/content/data"

LABEL_MAPS = {
    "wheat_primary": {
        "LeafBlight": ("wheat", "leaf_blight"),
        "WheatBlast": ("wheat", "wheat_blast"),
        "HealthyLeaf": ("wheat", "healthy"),
        "BlackPoint": ("wheat", "black_point"),
        "FusariumFootRot": ("wheat", "fusarium_foot_rot"),
    },
    "cotton_primary": {
        "fussarium_wilt": ("cotton", "fusarium_wilt"),
        "curl_virus": ("cotton", "curl_virus"),
        "healthy": ("cotton", "healthy"),
        "bacterial_blight": ("cotton", "bacterial_blight"),
    },
    "rice_primary": {
        "Tungro": ("rice", "tungro"),
        "Bacterialblight": ("rice", "bacterial_blight"),
        "Blast": ("rice", "blast"),
        "Brownspot": ("rice", "brown_spot"),
    },
    "sugarcane_primary": {
        "Yellow": ("sugarcane", "yellow"),
        "Mosaic": ("sugarcane", "mosaic"),
        "BacterialBlights": ("sugarcane", "bacterial_blight"),
        "Healthy": ("sugarcane", "healthy"),
        "RedRot": ("sugarcane", "red_rot"),
        "Rust": ("sugarcane", "rust"),
    },
    "citrus_primary": {
        "Black spot": ("citrus", "black_spot"),
        "Canker": ("citrus", "canker"),
        "Melanose": ("citrus", "melanose"),
        "Healthy": ("citrus", "healthy"),
        "Greening": ("citrus", "greening"),
    },
    "mango_primary": {
        "Powdery Mildew": ("mango", "powdery_mildew"),
        "Cutting Weevil": ("mango", "cutting_weevil"),
        "Anthracnose": ("mango", "anthracnose"),
        "Bacterial Canker": ("mango", "bacterial_canker"),
        "Sooty Mould": ("mango", "sooty_mould"),
        "Gall Midge": ("mango", "gall_midge"),
        "Healthy": ("mango", "healthy"),
        "Die Back": ("mango", "die_back"),
    },
    "multi_crop_supplement": {
        # Only entries for our 6 target crops — Maize/Corn folders in this
        # dataset are intentionally left unmapped (not a target crop).
        "RedRust sugarcane": ("sugarcane", "rust"),
        "Wheat___Yellow_Rust": ("wheat", "yellow_rust"),
        "Tungro": ("rice", "tungro"),
        "Wheat mite": ("wheat", "mite"),
        "Anthracnose on Cotton": ("cotton", "anthracnose"),
        "Healthy Wheat": ("wheat", "healthy"),
        "Cotton Aphid": ("cotton", "aphid"),
    },
}

def find_dirs_by_name(root, name):
    return [dirpath for dirpath, _, _ in os.walk(root) if os.path.basename(dirpath) == name]

def normalize_dataset(key, raw_path, label_map):
    count = 0
    for subfolder, (crop, disease) in label_map.items():
        src_dirs = find_dirs_by_name(raw_path, subfolder)
        if not src_dirs:
            print(f"  WARNING: no directory named '{subfolder}' found under {raw_path}, skipping")
            continue
        dst_dir = os.path.join(UNIFIED_ROOT, crop, disease)
        os.makedirs(dst_dir, exist_ok=True)
        for src_dir in src_dirs:
            for fname in os.listdir(src_dir):
                src_file = os.path.join(src_dir, fname)
                if not os.path.isfile(src_file):
                    continue
                dst_file = os.path.join(dst_dir, f"{key}_{fname}")
                if os.path.exists(dst_file):
                    base, ext = os.path.splitext(dst_file)
                    dst_file = f"{base}_{abs(hash(src_dir)) % 100000}{ext}"
                shutil.copyfile(src_file, dst_file)
                count += 1
    return count

for key in ["wheat_primary", "cotton_primary", "rice_primary",
            "sugarcane_primary", "citrus_primary", "mango_primary",
            "multi_crop_supplement"]:
    n = normalize_dataset(key, RAW_PATHS[key], LABEL_MAPS[key])
    print(f"{key}: copied {n} files")

In [ ]:
import shutil

UNIFIED_ROOT = "/content/data"

# Filled from Task 4's printed trees (2026-08-12 run). Two gaps still open:
#  - wheat_primary: class names not yet known — nested under Train/Test/Validation,
#    deeper than Task 4's max_depth=2 explore. Run the snippet in the next cell
#    to list them, then fill wheat_primary below.
#  - citrus_primary / multi_crop_supplement: only partially captured from a
#    truncated paste — the 5 citrus classes and 8 supplement classes below are
#    confirmed real, but there may be more not yet seen. Safe to proceed with
#    what's confirmed; add more entries later if you spot them.
LABEL_MAPS = {
    "wheat_primary": {
        # "<class_folder_name>": ("wheat", "<disease_label>"),
    },
    "cotton_primary": {
        "fussarium_wilt": ("cotton", "fusarium_wilt"),
        "curl_virus": ("cotton", "curl_virus"),
        "healthy": ("cotton", "healthy"),
        "bacterial_blight": ("cotton", "bacterial_blight"),
    },
    "rice_primary": {
        "Tungro": ("rice", "tungro"),
        "Bacterialblight": ("rice", "bacterial_blight"),
        "Blast": ("rice", "blast"),
        "Brownspot": ("rice", "brown_spot"),
    },
    "sugarcane_primary": {
        "Yellow": ("sugarcane", "yellow"),
        "Mosaic": ("sugarcane", "mosaic"),
        "BacterialBlights": ("sugarcane", "bacterial_blight"),
        "Healthy": ("sugarcane", "healthy"),
        "RedRot": ("sugarcane", "red_rot"),
        "Rust": ("sugarcane", "rust"),
    },
    "citrus_primary": {
        "Black spot": ("citrus", "black_spot"),
        "Canker": ("citrus", "canker"),
        "Melanose": ("citrus", "melanose"),
        "Healthy": ("citrus", "healthy"),
        "Greening": ("citrus", "greening"),
    },
    "mango_primary": {
        "Powdery Mildew": ("mango", "powdery_mildew"),
        "Cutting Weevil": ("mango", "cutting_weevil"),
        "Anthracnose": ("mango", "anthracnose"),
        "Bacterial Canker": ("mango", "bacterial_canker"),
        "Sooty Mould": ("mango", "sooty_mould"),
        "Gall Midge": ("mango", "gall_midge"),
        "Healthy": ("mango", "healthy"),
        "Die Back": ("mango", "die_back"),
    },
    "multi_crop_supplement": {
        # Only entries for our 6 target crops — Maize/Corn folders in this
        # dataset are intentionally left unmapped (not a target crop).
        "RedRust sugarcane": ("sugarcane", "rust"),
        "Wheat___Yellow_Rust": ("wheat", "yellow_rust"),
        "Tungro": ("rice", "tungro"),
        "Wheat mite": ("wheat", "mite"),
        "Anthracnose on Cotton": ("cotton", "anthracnose"),
        "Healthy Wheat": ("wheat", "healthy"),
        "Cotton Aphid": ("cotton", "aphid"),
        # Add more rows here if you spot other target-crop folders when you
        # scroll further through this dataset's Task 4 output (e.g. other
        # sugarcane/rice/citrus/mango classes may exist further down).
    },
}

def find_dirs_by_name(root, name):
    # Searches the whole tree, not just direct children — handles datasets
    # like wheat_primary where classes sit under Train/Test/Validation, and
    # merges all matches together (we don't keep the source's original split).
    return [dirpath for dirpath, _, _ in os.walk(root) if os.path.basename(dirpath) == name]

def normalize_dataset(key, raw_path, label_map):
    count = 0
    for subfolder, (crop, disease) in label_map.items():
        src_dirs = find_dirs_by_name(raw_path, subfolder)
        if not src_dirs:
            print(f"  WARNING: no directory named '{subfolder}' found under {raw_path}, skipping")
            continue
        dst_dir = os.path.join(UNIFIED_ROOT, crop, disease)
        os.makedirs(dst_dir, exist_ok=True)
        for src_dir in src_dirs:
            for fname in os.listdir(src_dir):
                src_file = os.path.join(src_dir, fname)
                if not os.path.isfile(src_file):
                    continue
                dst_file = os.path.join(dst_dir, f"{key}_{fname}")
                if os.path.exists(dst_file):
                    # same filename appears in more than one split (e.g. Train + Test) — disambiguate
                    base, ext = os.path.splitext(dst_file)
                    dst_file = f"{base}_{abs(hash(src_dir)) % 100000}{ext}"
                shutil.copyfile(src_file, dst_file)
                count += 1
    return count

for key in ["wheat_primary", "cotton_primary", "rice_primary",
            "sugarcane_primary", "citrus_primary", "mango_primary",
            "multi_crop_supplement"]:
    n = normalize_dataset(key, RAW_PATHS[key], LABEL_MAPS[key])
    print(f"{key}: copied {n} files")

A `0` count for any dataset means its `LABEL_MAPS` entry above is still empty or wrong for that dataset's actual folder names — fix and re-run before continuing.

Note: `leafnet_field` is intentionally excluded here — it feeds the domain-pretrain step in Notebook 2, not this per-crop manifest. It stays referenced via `RAW_PATHS["leafnet_field"]` only.

## Task 6: Build the manifest CSV

In [ ]:
import pandas as pd

rows = []
for crop in os.listdir(UNIFIED_ROOT):
    crop_dir = os.path.join(UNIFIED_ROOT, crop)
    if not os.path.isdir(crop_dir):
        continue
    for disease in os.listdir(crop_dir):
        disease_dir = os.path.join(crop_dir, disease)
        if not os.path.isdir(disease_dir):
            continue
        for fname in os.listdir(disease_dir):
            fpath = os.path.join(disease_dir, fname)
            if os.path.isfile(fpath):
                rows.append({
                    "filepath": fpath,
                    "crop": crop,
                    "disease": disease,
                    "is_field": False,  # lab-shot Kaggle sources; flip per-source if known field-shot
                })

manifest = pd.DataFrame(rows)
manifest.to_csv(os.path.join(UNIFIED_ROOT, "manifest.csv"), index=False)
print(manifest.shape)
print(manifest.groupby(["crop", "disease"]).size())

In [ ]:
assert manifest["crop"].nunique() == 6, f"Expected 6 crops, got {manifest['crop'].nunique()}: {manifest['crop'].unique()}"
print("All 6 crops present.")

## Task 7: Integrity checks — corrupt images, duplicates, class balance report

In [ ]:
import hashlib
from PIL import Image

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

manifest["valid"] = manifest["filepath"].apply(is_valid_image)
corrupt_count = (~manifest["valid"]).sum()
print(f"Corrupt/unreadable images: {corrupt_count}")

manifest = manifest[manifest["valid"]].drop(columns=["valid"])

manifest["hash"] = manifest["filepath"].apply(file_hash)
dup_count = manifest.duplicated(subset="hash").sum()
print(f"Duplicate images (by content hash): {dup_count}")

manifest = manifest.drop_duplicates(subset="hash").drop(columns=["hash"])

manifest.to_csv(os.path.join(UNIFIED_ROOT, "manifest.csv"), index=False)
print(f"Final manifest: {manifest.shape[0]} images across {manifest['crop'].nunique()} crops")
print(manifest.groupby(["crop", "disease"]).size())

In [ ]:
counts = manifest.groupby(["crop", "disease"]).size()
thin = counts[counts < 50]
if len(thin):
    print("WARNING — classes with <50 images (may need oversampling or more sourcing):")
    print(thin)
else:
    print("No classes below 50 images.")

import shutil

# Zip first — copying ~35k individual files through the Drive FUSE mount
# hits Google's per-user API request quota fast, regardless of free storage
# space. A single zip is one file transfer instead of tens of thousands.
ZIP_BASE = "/content/crop_disease_data"
shutil.make_archive(ZIP_BASE, "zip", UNIFIED_ROOT)
zip_path = ZIP_BASE + ".zip"
print(f"Zipped to {zip_path}, size: {os.path.getsize(zip_path) / 1e6:.1f} MB")

DRIVE_DEST_DIR = "/content/drive/MyDrive/crop_disease"
os.makedirs(DRIVE_DEST_DIR, exist_ok=True)
shutil.copyfile(zip_path, os.path.join(DRIVE_DEST_DIR, "data.zip"))
print(f"Persisted to {DRIVE_DEST_DIR}/data.zip")

In [ ]:
import zipfile

drive_zip = os.path.join(DRIVE_DEST_DIR, "data.zip")
assert os.path.exists(drive_zip), "Zip not found on Drive"
with zipfile.ZipFile(drive_zip) as zf:
    names = zf.namelist()
assert any(n.endswith("manifest.csv") for n in names), "manifest.csv missing from zip"
print(f"Drive zip verified: {len(names)} files, {os.path.getsize(drive_zip) / 1e6:.1f} MB")

In [ ]:
saved_manifest = pd.read_csv(os.path.join(DRIVE_DEST, "manifest.csv"))
assert len(saved_manifest) == len(manifest), "Row count mismatch after copy to Drive"
print("Drive copy verified:", saved_manifest.shape)